# 02 — Análise Exploratória de Dados (EDA)

Este notebook realiza a análise exploratória do dataset de churn de clientes,
incluindo estatísticas descritivas, distribuições e correlações.

**Pré-requisito:** Execute `01_coleta_dados.ipynb` ou `scripts/01_coleta_dados.py` antes.

---

## 2.1 Importações e Configurações

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de visualização
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 6), "font.size": 11})
sns.set_theme(style="whitegrid", palette="muted")

CAMINHO_DADOS = os.path.join("..", "dados", "bruto", "dataset_clientes.csv")
DIR_FIGURAS = os.path.join("..", "resultados", "figuras")
DIR_METRICAS = os.path.join("..", "resultados", "metricas")
os.makedirs(DIR_FIGURAS, exist_ok=True)
os.makedirs(DIR_METRICAS, exist_ok=True)

## 2.2 Carregamento e Visão Geral dos Dados

In [ ]:
df = pd.read_csv(CAMINHO_DADOS)
print(f"Shape: {df.shape}")
print(f"\nTipos de dados:\n{df.dtypes}")
df.head(10)

In [ ]:
# Informações gerais
df.info()

In [ ]:
# Valores ausentes
ausentes = df.isnull().sum()
ausentes_pct = (ausentes / len(df) * 100).round(2)
pd.DataFrame({"Ausentes": ausentes, "% Ausentes": ausentes_pct}).query("Ausentes > 0")

## 2.3 Estatísticas Descritivas

In [ ]:
colunas_num = ["idade", "renda_mensal", "tempo_conta_meses",
               "num_produtos", "saldo", "num_reclamacoes", "satisfacao"]

stats = df[colunas_num].describe().T
stats["mediana"] = df[colunas_num].median()
stats["cv"] = (stats["std"] / stats["mean"] * 100).round(2)
stats

In [ ]:
# Salvar estatísticas descritivas
stats.to_csv(os.path.join(DIR_METRICAS, "estatisticas_descritivas.csv"))
print("✓ Estatísticas descritivas salvas.")

## 2.4 Distribuição da Variável-alvo (Churn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem
df["churn"].value_counts().plot(kind="bar", ax=axes[0], color=["#3498db", "#e74c3c"],
                                 edgecolor="white")
axes[0].set_title("Contagem de Churn", fontweight="bold")
axes[0].set_xticklabels(["Não-Churn (0)", "Churn (1)"], rotation=0)

# Proporção
df["churn"].value_counts(normalize=True).plot(kind="pie", ax=axes[1],
    autopct="%1.1f%%", colors=["#3498db", "#e74c3c"], startangle=90)
axes[1].set_title("Proporção de Churn", fontweight="bold")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 2.5 Distribuições das Variáveis Numéricas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Distribuição das Variáveis por Status de Churn", fontsize=14, fontweight="bold")

colunas = ["idade", "renda_mensal", "tempo_conta_meses", "saldo", "num_reclamacoes", "satisfacao"]

for ax, col in zip(axes.ravel(), colunas):
    for churn_val, cor, label in [(0, "#3498db", "Não-Churn"), (1, "#e74c3c", "Churn")]:
        subset = df[df["churn"] == churn_val][col].dropna()
        ax.hist(subset, bins=25, alpha=0.6, color=cor, label=label, edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(os.path.join(DIR_FIGURAS, "distribuicao_features.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2.6 Matriz de Correlação

In [ ]:
colunas_corr = df.select_dtypes(include=[np.number]).columns.tolist()
colunas_corr = [c for c in colunas_corr if c != "id_cliente"]
corr = df[colunas_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Matriz de Correlação de Pearson", fontsize=14, fontweight="bold", pad=15)

plt.tight_layout()
fig.savefig(os.path.join(DIR_FIGURAS, "correlacao_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2.7 Taxa de Churn por Variável Categórica

In [ ]:
colunas_cat = ["sexo", "canal_aquisicao", "tem_credito", "ativo"]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Taxa de Churn por Variável Categórica", fontsize=14, fontweight="bold")

for ax, col in zip(axes.ravel(), colunas_cat):
    taxa = df.groupby(col)["churn"].mean().sort_values(ascending=False)
    bars = ax.bar(taxa.index.astype(str), taxa.values, color="#2ecc71", edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.set_ylabel("Taxa de Churn")
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, taxa.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.1%}", ha="center", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(os.path.join(DIR_FIGURAS, "churn_por_categoria.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2.8 Conclusões da EDA

- **Desbalanceamento:** A variável-alvo apresenta desbalanceamento moderado.
- **Valores ausentes:** `renda_mensal` (~5%) e `satisfacao` (~3%) possuem dados faltantes.
- **Correlações:** Verificar variáveis com correlação significativa com `churn`.
- **Próximo passo:** Pré-processamento (notebook 03).